<h1 style="color:#C2185B">Isolation Forest</h1>

<h2 style="color:#C2185B">Owner</h2>

<h3 style="color:#C2185B">Goal</h3>

Fit and tune Isolation Forest on the cleaned data, log every experiment, and save final anomaly scores for evaluation in 05.

<h4 style="color:#C2185B">Outline</h4>

*Feel free to expand on anything below, this is just a starting skeleton, not a strict limit on what you explore or add.*

- **[I. Configuration & Setup](#config-setup)**
- **[II. Data Loading and Quality Check](#data-loading-quality-check)**
  1. [Load cleaned features and label](#load-cleaned-data)
  2. [Confirm shapes match what 01 produced](#confirm-shapes)
  3. [Confirm no missing values, isFraud only in label](#confirm-no-missing)
- **[III. Baseline Experiment](#baseline-experiment)**
  1. [Fit baseline Isolation Forest](#fit-baseline)
  2. [Label-based sanity check](#baseline-sanity-check)
  3. [Baseline runtime](#baseline-runtime)
- **[IV. Hyperparameter Experiments](#hyperparameter-experiments)**
  1. [Contamination sweep](#contamination-sweep)
  2. [n_estimators sweep](#n-estimators-sweep)
  3. [max_samples (psi) sweep](#max-samples-sweep)
  4. [Runtime vs. parameter tradeoffs](#runtime-tradeoffs)
- **[V. Final Model](#final-model)**
  1. [Choose final hyperparameters](#choose-final-hyperparameters)
  2. [Fit final model and generate final anomaly scores](#fit-final-model)
- **[VI. Save Results](#save-results)**
  1. [Save anomaly scores to results/isolation_forest/](#save-scores)
- **[VII. Sanity Check](#sanity-check)**

<a id="config-setup"></a>
<h1 style="color:#C2185B">I. Configuration & Setup</h1>

In [ ]:
# Import the libraries needed for this notebook
import os
import time
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

# Define base directory for the project: walk up from the current working
# directory until a file that only exists at the repo root is found.
PROJECT_MARKERS = ('.git', 'requirements.txt')

def find_project_root(start_dir=None, markers=PROJECT_MARKERS):
    # Start looking from the current working directory, unless a different
    # start_dir was given
    current = os.path.abspath(start_dir or os.getcwd())
    while True:
        # If any marker file exists in this directory, this is the project root
        if any(os.path.exists(os.path.join(current, marker)) for marker in markers):
            return current
        # Otherwise, move up one directory and keep looking
        parent = os.path.dirname(current)
        if parent == current:
            # Reached the filesystem root without finding a marker, stop
            raise FileNotFoundError(f"Could not locate project root (looked for {markers})")
        current = parent

# Find the project root directory
BASE_DIR = find_project_root()

# Define the path to the data directory
DATA_DIR = os.path.join(BASE_DIR, 'data')
CLEANED_DATA_DIR = os.path.join(DATA_DIR, 'cleaned')

# Define the path to this method's results directory, and create it if it
# doesn't exist yet
RESULTS_DIR = os.path.join(BASE_DIR, 'results', 'isolation_forest')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Print the resolved paths, to confirm everything points where expected
print(f"Project root directory: {BASE_DIR}")
print(f"Cleaned data directory: {CLEANED_DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")

In [ ]:
# Experiment log: every experiment below should call log_experiment(...) so
# nothing gets lost between cells, and so there's a record for the report and
# the presentation's "Fine-Tuning Values Tried" appendix slide.

# Define the path to the experiment log file
EXPERIMENT_LOG_PATH = os.path.join(RESULTS_DIR, 'experiment_log.csv')

def log_experiment(name, params, metrics):
    """Append one experiment's parameters and metrics as a row, and save it."""
    # Combine the experiment name, a timestamp, and the given params/metrics
    # into a single row
    row = {
        'experiment': name,
        'timestamp': pd.Timestamp.now().isoformat(),
        **params,
        **metrics,
    }
    # Turn that row into a one-row DataFrame so it can be appended to the CSV
    row_df = pd.DataFrame([row])

    # Only write the header the first time the file is created
    write_header = not os.path.exists(EXPERIMENT_LOG_PATH)

    # Append this row to the log file (or create it, if this is the first call)
    row_df.to_csv(EXPERIMENT_LOG_PATH, mode='a', header=write_header, index=False)

    print(f"✅ Logged '{name}' to {os.path.basename(EXPERIMENT_LOG_PATH)}")
    return row

# Print where the experiment log lives
print(f"Experiment log: {EXPERIMENT_LOG_PATH}")

<a id="data-loading-quality-check"></a>
<h1 style="color:#C2185B">II. Data Loading and Quality Check</h1>

<a id="load-cleaned-data"></a>
<h2 style="color:#C2185B">1. Load cleaned features and label</h2>

In [ ]:
# Load the cleaned features and label saved by 01-data-preparation.ipynb

# Define the paths to the cleaned features and label files
features_path = os.path.join(CLEANED_DATA_DIR, 'features.parquet')
label_path = os.path.join(CLEANED_DATA_DIR, 'label.parquet')

# Check if the required files exist before loading
required_paths = {
    "features": features_path,
    "label": label_path,
}
missing = [name for name, path in required_paths.items() if not os.path.exists(path)]

if missing:
    # Warn for every missing file, rather than failing on just the first one
    for name in missing:
        print(f"⚠️ Missing {name} data. Run 01-data-preparation.ipynb first to generate it.")
else:
    # Load the features and label, now that both files are confirmed to exist
    features = pd.read_parquet(features_path)
    label = pd.read_parquet(label_path)['isFraud']
    print(f"✅ Loaded cleaned data: {features.shape[0]} rows, {features.shape[1]} feature columns.")

**Comment:**

<a id="confirm-shapes"></a>
<h2 style="color:#C2185B">2. Confirm shapes match what 01 produced</h2>

In [ ]:
# Confirm the loaded shapes match what 01-data-preparation.ipynb reported
# Example: compare features.shape and label.shape against the row/column
# counts 01 printed when it saved them
# Goal: catch a mismatch early if the wrong file got loaded, or if 01 was
# rerun with different cleaning decisions since.

**Comment:**

<a id="confirm-no-missing"></a>
<h2 style="color:#C2185B">3. Confirm no missing values, isFraud only in label</h2>

In [ ]:
# Confirm the loaded data is still clean
# Example: features.isnull().sum().sum() should be 0; 'isFraud' should not
# be a column in features at all
# Goal: catch it immediately if the cleaned files got corrupted or
# regenerated incorrectly, before fitting anything on top of them.

**Comment:**

<a id="baseline-experiment"></a>
<h1 style="color:#C2185B">III. Baseline Experiment</h1>

<a id="fit-baseline"></a>
<h2 style="color:#C2185B">1. Fit baseline Isolation Forest</h2>

In [ ]:
# Fit a baseline Isolation Forest with default parameters
# Example: IsolationForest(contamination='auto', random_state=42).fit(features)
# Log it: once you have scores, call log_experiment('baseline', params={...},
# metrics={...}) including precision, recall, roc_auc, pr_auc, and accuracy
# (same metrics used everywhere else in this notebook), so it's saved to the
# experiment log, not just printed and lost
# Goal: a reference point every later experiment gets compared against.

**Comment:**

<a id="baseline-sanity-check"></a>
<h2 style="color:#C2185B">2. Label-based sanity check</h2>

In [ ]:
# Check the baseline's anomaly scores against the real labels
# "Sanity check" means a quick, informal check, not a full evaluation, just
# confirming the scores are picking up something real before investing time
# in tuning. If this looks no better than random, something's likely broken
# (features, the merge, a bug in scoring), worth stopping and investigating
# here rather than tuning a broken baseline.
# Example: fraud rate in the top 10 and top k scored transactions (same
# pattern as Task 4), plus ROC-AUC and PR-AUC computed directly from the
# continuous anomaly score (roc_auc_score(label, anomaly_score),
# average_precision_score(label, anomaly_score)), these don't need a
# contamination threshold at all
# Goal: a first read on whether the baseline is picking up any real signal
# at all, before tuning anything.

**Comment:**

<a id="baseline-runtime"></a>
<h2 style="color:#C2185B">3. Baseline runtime</h2>

In [ ]:
# Time how long the baseline fit took
# Example: wrap the .fit() call with time.time() before and after
# Log it: include the runtime in the metrics passed to log_experiment for
# the baseline (go back and re-log it here if you didn't capture it above)
# Goal: a reference runtime for RQ3 (computational cost), to compare every
# later experiment against.

**Comment:**

<a id="hyperparameter-experiments"></a>
<h1 style="color:#C2185B">IV. Hyperparameter Experiments</h1>

<a id="contamination-sweep"></a>
<h2 style="color:#C2185B">1. Contamination sweep</h2>

In [ ]:
# Try several contamination values (e.g. 0.01, 0.02, 0.035, 0.05, 0.10),
# fitting a separate model for each
#
# Worked example for ONE value below (contamination=0.035, close to the
# real fraud rate). Follow the same pattern in a loop for the other values.

from sklearn.metrics import precision_score, recall_score, roc_auc_score, average_precision_score, accuracy_score

# Define the contamination value
contamination_value = 0.035

# Start timing the model fitting
start_time = time.time()

# Define and fit the Isolation Forest model with the specified contamination value
model = IsolationForest(contamination=contamination_value, n_estimators=100, max_samples=256, random_state=42)
model.fit(features)

# Calculate the runtime of the model fitting
runtime = time.time() - start_time

# Calculate the anomaly scores and predictions
anomaly_score = -model.score_samples(features)  # higher = more anomalous

# Predict anomalies: -1 = anomaly, 1 = normal
predictions = model.predict(features)  # -1 = anomaly, 1 = normal

# Calculate the predicted fraud labels based on the predictions
predicted_fraud = (predictions == -1).astype(int)

# Calculate evaluation metrics
precision = precision_score(label, predicted_fraud)
recall = recall_score(label, predicted_fraud)
roc_auc = roc_auc_score(label, anomaly_score)
pr_auc = average_precision_score(label, anomaly_score)
accuracy = accuracy_score(label, predicted_fraud)
# Note: accuracy is misleading here, fraud is ~3.5% of the data, so a model
# that predicts "never fraud" scores ~96.5% accuracy while catching nothing.
# Log it anyway for completeness, but don't read much into it, ROC-AUC and
# PR-AUC are the metrics that actually matter on imbalanced data like this.

# Log the experiment results
log_experiment(
    name=f"contamination_{contamination_value}",
    params={"contamination": contamination_value, "n_estimators": 100, "max_samples": 256},
    metrics={
        "precision": precision,
        "recall": recall,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "accuracy": accuracy,
        "runtime_seconds": runtime,
    },
)

# Next: loop over the other values (e.g. [0.01, 0.02, 0.05, 0.10]) the same
# way, calling log_experiment once per value, so every value ends up as its
# own row in experiment_log.csv.

**Comment:**

<a id="n-estimators-sweep"></a>
<h2 style="color:#C2185B">2. n_estimators sweep</h2>

In [ ]:
# Try several n_estimators values (e.g. 50, 100, 200, 500)
# Example: loop over the values, fit, compute ROC-AUC and PR-AUC directly
# from each one's continuous anomaly score (these don't need a contamination
# threshold, so this sweep can stay label-light the same way); you can also
# compare rankings directly against each other (e.g. with
# scipy.stats.spearmanr) to see when results stabilize
# Log it: call log_experiment(...) for each value, including roc_auc, pr_auc,
# and runtime
# Goal: find the smallest n_estimators that still gives stable results, more
# trees cost more time for diminishing returns.

**Comment:**

<a id="max-samples-sweep"></a>
<h2 style="color:#C2185B">3. max_samples (psi) sweep</h2>

In [ ]:
# Try several max_samples values (e.g. 128, 256, 512, 10000, the full dataset)
# Example: loop over the values, fit, compute ROC-AUC and PR-AUC from each
# one's continuous anomaly score, and compare frauds caught in the top k
# against each other
# Log it: call log_experiment(...) for each value, including roc_auc, pr_auc
# Goal: the paper recommends a small psi (256); check whether that actually
# holds up on this dataset, or whether a larger sample does better here.

**Comment:**

<a id="runtime-tradeoffs"></a>
<h2 style="color:#C2185B">4. Runtime vs. parameter tradeoffs</h2>

In [ ]:
# Pull the runtimes logged above together into one comparison
# Example: read experiment_log.csv back in, plot runtime vs. each swept
# parameter
# Goal: a direct answer to RQ3, how does cost scale with each hyperparameter?

**Comment:**

<a id="final-model"></a>
<h1 style="color:#C2185B">V. Final Model</h1>

<a id="choose-final-hyperparameters"></a>
<h2 style="color:#C2185B">1. Choose final hyperparameters</h2>

In [ ]:
# Based on all experiments above, pick the final contamination, n_estimators,
# and max_samples to use
# Goal: a single, clearly justified set of values; write down why, pointing
# back at the logged experiments that led to this choice.

**Comment:**

<a id="fit-final-model"></a>
<h2 style="color:#C2185B">2. Fit final model and generate final anomaly scores</h2>

In [ ]:
# Fit the final Isolation Forest with the chosen hyperparameters on the full
# feature set, and compute anomaly scores for every row
# Log it: call log_experiment('final', params={...}, metrics={...}), including
# precision, recall, roc_auc, pr_auc, accuracy, and runtime, the same metrics
# used in the sweeps above, so the final choice is directly comparable to
# everything that came before it
# Goal: the actual scores that get saved and handed to 05-evaluation.ipynb.

**Comment:**

<a id="save-results"></a>
<h1 style="color:#C2185B">VI. Save Results</h1>

<a id="save-scores"></a>
<h2 style="color:#C2185B">1. Save anomaly scores to results/isolation_forest/</h2>

In [ ]:
# Save the final anomaly scores to results/isolation_forest/
# Example: a DataFrame with a row identifier and anomaly_score, saved with
# .to_parquet()
# Goal: a single source of Isolation Forest's output that
# 05-evaluation.ipynb loads from.

**Comment:**

<a id="sanity-check"></a>
<h1 style="color:#C2185B">VII. Sanity Check</h1>

Before considering this notebook done, confirm all of the following:

1. Make sure `experiment_log.csv` actually has a row for every experiment you ran.
2. Make sure the final model's hyperparameters match what V.1 decided, and that the decision is written down.
3. Make sure the saved anomaly scores file has one score per row, no missing values, and the same row count as `features`.
4. Make sure the runtime comparison (IV.4) actually reflects the logged numbers.